1\. Preprocessing Pipeline Explanation
--------------------------------------

This script implements a comprehensive data preprocessing pipeline designed to prepare the mlibre/palestine dataset from Hugging Face for fine-tuning a Large Language Model (LLM), specifically google/gemma-2b-it.

The primary goal is to transform raw textual data into a structured instruction-response format suitable for supervised fine-tuning, while also ensuring that the data conforms to the LLM's context window limits.

### I. Configuration and Setup

*   **MODEL\_ID**: Specifies the pre-trained model (google/gemma-2b-it) used for tokenizer alignment. This ensures that the tokenization process accurately reflects the model's understanding of text.
    
*   **MAX\_LENGTH**: Defines the maximum sequence length (1024 tokens) for the LLM's context window. Data exceeding this limit will be skipped to optimize fine-tuning on resource-constrained environments like free Colab GPUs.
    
*   **token**: Retrieves a Hugging Face authentication token from Colab's user data, necessary for accessing the model and tokenizer.
    

### II. Data Loading

*   The script directly loads the mlibre/palestine dataset from Hugging Face using datasets.load\_dataset.
    
*   The loaded dataset, initially a Hugging Face Dataset object, is converted into a pandas DataFrame for easier manipulation and processing.
    

### III. Data Cleaning (clean\_text function)

*   This function normalizes and cleans raw text data by:
    
    *   **Removing HTML tags**:to eliminate web-scraping artifacts.
        
    *   **Normalizing whitespace**: Consolidating tabs and multiple newlines into single spaces and stripping leading/trailing whitespace.
        
    *   **Fixing Scraper Artifacts**: Identifying and formatting patterns like "Answer 1..." to ensure the model distinguishes between different logical segments of the text.
        

### IV. Instruction Formatting (format\_instruction\_response function)

*   **Heuristic Mapping**: This function analyzes metadata (URLs and Page Titles) to determine if a text is a legal rebuttal, a news analysis, or a general summary.
    
*   **Instruction Injection**: It programmatically wraps the text in a prompt, such as _"Analyze the following from a Palestinian resistance perspective,"_ which guides the model's ideological behavior.
    

### V. Context Window Audit & Tokenization

*   The script uses the actual Gemma tokenizer to encode every prompt-response pair.
    
*   **Sequence Verification**: It calculates the exact token\_count. If a pair is too long for the GPU's memory (VRAM), it is discarded to prevent training crashes, rather than being truncated (which would lose the "Response" part of the data).
    

### VI. Train/Test Splitting

*   The final dataset is split (90% training, 10% testing) using train\_test\_split.
    
*   **Validation Strategy**: This ensures we have a "held-out" set of revolutionary text to test the model's performance on data it has never seen before.

In [ ]:
import pandas as pd
import json
import re
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

# ==========================================
# 1. CONFIGURATION & SETUP
# ==========================================
# Alignment: We update this to Gemma-2b-it to match our fine-tuning script.
# This ensures the 'Context Window Audit' uses the correct tokenizer logic.
MODEL_ID = "google/gemma-2b-it"
MAX_LENGTH = 1024  # Context window limit for efficient fine-tuning on free Colab GPU

from google.colab import userdata
token = userdata.get('Gamma')

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, token=token)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=token)

def clean_text(text):
    """
    Normalization: Removes web-scraping noise while preserving
    revolutionary/specific terminology.
    """
    if not isinstance(text, str):
        return ""

    # 1. Remove HTML artifacts if any exist
    text = re.sub(r'<[^>]+>', '', text)

    # 2. Normalize whitespace (remove excessive tabs/newlines)
    text = re.sub(r'\s+', ' ', text).strip()

    # 3. Handle specific formatting issues from the dataset
    # e.g., "Answer 1..." sometimes runs together with text
    text = re.sub(r'(Answer \d)', r'\n\1: ', text)

    return text

def format_instruction_response(row):
    """
    Formatting: Converts raw data into a structured Instruction-Response pair.
    Strategy depends on the source type found in 'pageTitle' or 'url'.
    """
    # Adjusting keys to match Hugging Face dataset structure
    text = row.get('text', '')
    title = row.get('articleTitle', '') or row.get('title', '')

    # Metadata in HF dataset might be nested or named differently
    source_title = str(row.get('pageTitle', '')).lower()
    url = str(row.get('url', '')).lower()

    # Fallback if metadata is missing but text exists
    if not source_title and 'metadata' in row and isinstance(row['metadata'], dict):
         source_title = str(row['metadata'].get('pageTitle', '')).lower()
         url = str(row['metadata'].get('url', '')).lower()

    instruction = ""
    response = ""

    # CASE A: Gold Standard Q&A (PaliAnswers)
    # Identified by 'Pali Answers' in pageTitle or 'rebuttal' in URL
    if 'pali answers' in source_title or 'rebuttal' in url:
        # Cleaning the claim title (removing emojis often found in PaliAnswers)
        clean_title = re.sub(r'[^\w\s\?]', '', str(title)).strip()

        instruction = f"Provide a de-colonial rebuttal to the claim: '{clean_title}'"
        response = text

    # CASE B: Articles/Analysis (Mondoweiss, Electronic Intifada, BDS)
    else:
        # Heuristic: If we have a title, use it to frame the instruction.
        if title and len(str(title)) > 10:
            instruction = f"Analyze the following topic from a Palestinian resistance perspective: {title}"
            response = text
        else:
            # Fallback if no clear title: General summarization instruction
            instruction = "Summarize the key material conditions described in the following text:"
            response = text

    # Final check to ensure we don't have empty pairs
    if len(instruction) < 10 or len(str(response)) < 20:
        return None

    return {
        "instruction": clean_text(instruction),
        "input": "", # Optional context field
        "output": clean_text(response)
    }

def process_and_tokenize():
    """
    Main Pipeline: Loads data from Hugging Face, cleans, formats, and checks token limits.
    """
    print("--- Starting Comprehensive Preprocessing Pipeline ---")

    # 1. Load the Data directly from Hugging Face
    try:
        print(f"Loading dataset 'mlibre/palestine' from Hugging Face...")
        dataset = load_dataset("mlibre/palestine", split="train")

        # Convert to pandas DataFrame for easier manipulation
        df = pd.DataFrame(dataset)
        print(f"Loaded {len(df)} raw records from Hugging Face.")

    except Exception as e:
        print(f"Error loading dataset from Hugging Face: {e}")
        return

    # 2. Apply Formatting & Cleaning
    print("Formatting Instruction-Response pairs...")
    formatted_data = []

    for index, row in df.iterrows():
        processed_entry = format_instruction_response(row)
        if processed_entry:
            formatted_data.append(processed_entry)

    print(f"Generated {len(formatted_data)} valid instruction pairs.")

    # 3. Tokenization & Context Window Check
    print(f"Loading tokenizer ({MODEL_ID}) to check sequence lengths...")
    try:
        # Gemma requires the 'google/gemma-2b-it' identifier
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    except Exception as e:
        print(f"Tokenizer load failed ({e}). Using simple length check as fallback.")
        tokenizer = None

    valid_dataset = []
    skipped_count = 0

    for entry in formatted_data:
        # Create the full prompt string to test length
        # Standard Alpaca/Llama prompt format
        full_prompt = f"### Instruction:\n{entry['instruction']}\n\n### Response:\n{entry['output']}"

        if tokenizer:
            token_count = len(tokenizer.encode(full_prompt))
            if token_count <= MAX_LENGTH:
                valid_dataset.append(entry)
            else:
                skipped_count += 1
        else:
            # Fallback: Approx 4 chars per token
            if len(full_prompt) / 4 <= MAX_LENGTH:
                valid_dataset.append(entry)

    print(f"Context Window Audit: Kept {len(valid_dataset)} pairs. Skipped {skipped_count} for exceeding {MAX_LENGTH} tokens.")

    # 4. Train/Test Split (Important for Evaluation Metrics)
    print("Splitting into Train/Test sets...")
    if len(valid_dataset) > 0:
        train_data, test_data = train_test_split(valid_dataset, test_size=0.1, random_state=42)

        # 5. Save to JSONL (Standard format for LLM fine-tuning libraries)
        def save_jsonl(data, filename):
            with open(filename, 'w', encoding='utf-8') as f:
                for entry in data:
                    json.dump(entry, f)
                    f.write('\n')

        save_jsonl(train_data, "train_dataset.jsonl")
        save_jsonl(test_data, "test_dataset.jsonl")

        print(f"\n--- Preprocessing Complete ---")
        print(f"Training Samples: {len(train_data)}")
        print(f"Testing Samples: {len(test_data)}")
        print("Files saved: 'train_dataset.jsonl', 'test_dataset.jsonl'")
    else:
        print("Error: No valid data pairs remained after processing.")

if __name__ == "__main__":
    process_and_tokenize()

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

--- Starting Comprehensive Preprocessing Pipeline ---
Loading dataset 'mlibre/palestine' from Hugging Face...
Loaded 7151 raw records from Hugging Face.
Formatting Instruction-Response pairs...
Generated 7151 valid instruction pairs.
Loading tokenizer (google/gemma-2b-it) to check sequence lengths...
Context Window Audit: Kept 2581 pairs. Skipped 4570 for exceeding 1024 tokens.
Splitting into Train/Test sets...

--- Preprocessing Complete ---
Training Samples: 2322
Testing Samples: 259
Files saved: 'train_dataset.jsonl', 'test_dataset.jsonl'
